# tSCS EMG — single pulse, tendon vibration (P04, 16-07-2026)

## What a single-pulse file is
One **recruitment sweep**: one stimulation window per intensity (10 → 100 mA in 10 mA steps),
each EMG channel stored as a ±100 ms snippet time-locked to the pulse (t = 0).

## What is measured
- **Latency** — picked **by hand** for every muscle × intensity: the time (ms after the pulse) where
  the response starts. `NaN` = no response. This is the one manual step; picks are saved to
  `results/latency_<file>.csv` and reused from there.
- **Peak-to-peak** — from each picked latency, the window
  `[latency + OFFSET_MS, latency + OFFSET_MS + WINDOW_MS]`; p2p = max − min in it (mV).

## Files — tendon vibration session (`tendonvibr.xlsx`)
Vibrator on the **right flexor tendon**; the **left arm is the within-subject control**.
Electrode 2 throughout. Three single-pulse sweeps:

| block | file | time | max |
|---|---|---|---|
| **Baseline** (vibrator not attached) | `141131` | 14:11 | 90 mA |
| **VIB OFF** (vibrator on the tendon, switched off) | `143516` | 14:35 | 80 mA |
| **VIB ON – flexors** | `150838` | 15:08 | 60 mA |

`L_DELmed` was not recorded that day (14 muscles). Each file needs its latencies picked once (§2);
Baseline already has them. The thing to look for is a **right-vs-left asymmetry that appears
only in VIB ON**.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import numpy as np
import matplotlib.pyplot as plt

from functions import (set_style, load_run, pretty, waterfall, waterfall_overlay,
                       latency_picker, save_latency_csv, load_latency_csv)
from functions.quantify import plot_p2p_markers
from functions.average import compare_per_muscle
from functions.compare import compare_p2p_violin, compare_latency_violin
set_style()


## 1 · Config

In [ ]:
D = "tSCS_CHUV_data/16-07-2026/P04tscsHealthy/"
CONDITIONS = [   # (label, file) in the order they are plotted: gray, orange, blue
    ("Baseline",       "Single_Pulse_autosave_20260716_141131_534ms.csv"),
    ("VIB OFF",        "Single_Pulse_autosave_20260716_143516_906ms.csv"),
    ("VIB ON flexors", "Single_Pulse_autosave_20260716_150838_524ms.csv"),
]
USE = [0, 1, 2]        # <-- which conditions to compare (indices into CONDITIONS)

LABELS = [CONDITIONS[i][0] for i in USE]
CSVS   = [D + CONDITIONS[i][1] for i in USE]

XLIM      = (-20, 80)   # ms shown in the waterfalls / picker
WINDOW_MS = 20.0        # peak-to-peak window length (ms)
OFFSET_MS = 2.0         # ...starting this long after the picked latency (ms)
YLIM_P2P  = (-0.5, 10)  # y-range for the pooled p2p comparison; None = auto
YLIM_LAT  = (0, 30)     # y-range for latency plots (ms)

RUNS    = [load_run(f) for f in CSVS]
muscles = [c for c in RUNS[0][2] if c != "Trigger A" and all(c in r[2] for r in RUNS)]
for lab, (meta_, t_, sig_) in zip(LABELS, RUNS):
    print(f"{lab:16s} electrode {meta_[0]['electrode']} | pw {meta_[0]['pw_us']} us | "
          f"intensities {[x['amp_ma'] for x in meta_]} mA")

def _latfile(csv):
    return f"results/latency_{os.path.splitext(os.path.basename(csv))[0]}.csv"
for lab, csv in zip(LABELS, CSVS):
    f = _latfile(csv)
    print(f"{lab:16s} " + ("picks saved:  " if os.path.exists(f) else "NO PICKS YET: ") + f)


## 2 · Manual latency picking — one block per condition, do each ONCE

Every figure below reads the picks from `results/latency_<file>.csv`; the config cell above says
which conditions still say **NO PICKS YET**. Each condition has its own three cells:

- **A** — loads that file (and any picks already saved for it). Always safe to run.
- **B** — the picker: **uncomment**, run, click the response onset on every muscle × intensity
  (**Set NaN** = no response · ◀ Prev / Next ▶ · muscle dropdown), then comment it again.
- **C** — saves the picks to that file's own CSV: **uncomment**, run, comment again.

The cells of one block only talk to each other (`picks_<name>`), so running the blocks in any
order, or the rest of the notebook in between, can't mix conditions up. A save is refused if
the picks don't match the file's intensities.

### 2·Baseline

In [ ]:
# ---- Baseline · A: load ----
CSV_base = D + CONDITIONS[0][1]
meta_base, t_base, sig_base = load_run(CSV_base)
picks_base = load_latency_csv(_latfile(CSV_base)) if os.path.exists(_latfile(CSV_base)) else None
print("Baseline:", CSV_base.split("/")[-1], "|", [m["amp_ma"] for m in meta_base], "mA")
print("existing picks reloaded - continue/correct them" if picks_base else "no picks yet - pick them in B")


In [ ]:
# # ---- Baseline · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_base = latency_picker(meta_base, t_base, sig_base, muscles, xlim=XLIM, manual_peaks=picks_base)


In [ ]:
# # ---- Baseline · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); 
# %matplotlib inline
# print("saved", save_latency_csv(picks_base, muscles, CSV_base, meta=meta_base))


### 2·VIB OFF

In [ ]:
# ---- VIB OFF · A: load ----
CSV_off = D + CONDITIONS[1][1]
meta_off, t_off, sig_off = load_run(CSV_off)
picks_off = load_latency_csv(_latfile(CSV_off)) if os.path.exists(_latfile(CSV_off)) else None
print("VIB OFF:", CSV_off.split("/")[-1], "|", [m["amp_ma"] for m in meta_off], "mA")
print("existing picks reloaded - continue/correct them" if picks_off else "no picks yet - pick them in B")


In [ ]:
# ---- VIB OFF · B: PICK (uncomment, run, click through, comment again) ----
%matplotlib widget
picks_off = latency_picker(meta_off, t_off, sig_off, muscles, xlim=XLIM, manual_peaks=picks_off)


In [ ]:
# # ---- VIB OFF · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); 
# %matplotlib inline
# print("saved", save_latency_csv(picks_off, muscles, CSV_off, meta=meta_off))


### 2·VIB ON flexors

In [ ]:
# ---- VIB ON flexors · A: load ----
CSV_on = D + CONDITIONS[2][1]
meta_on, t_on, sig_on = load_run(CSV_on)
picks_on = load_latency_csv(_latfile(CSV_on)) if os.path.exists(_latfile(CSV_on)) else None
print("VIB ON flexors:", CSV_on.split("/")[-1], "|", [m["amp_ma"] for m in meta_on], "mA")
print("existing picks reloaded - continue/correct them" if picks_on else "no picks yet - pick them in B")


In [ ]:
# # ---- VIB ON flexors · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_on = latency_picker(meta_on, t_on, sig_on, muscles, xlim=XLIM, manual_peaks=picks_on)


In [ ]:
# # ---- VIB ON flexors · C: SAVE (uncomment, run, comment again) ----
# plt.close("all");
# %matplotlib inline
# print("saved", save_latency_csv(picks_on, muscles, CSV_on, meta=meta_on))


## 3 · Raw traces — waterfalls

One trace per intensity stacked at its amplitude, red = stimulus artifact. **Same gain per muscle
in both** (the first call returns the gains, the second reuses them), so a smaller response
with lidocaine actually draws smaller.

### 3a · First condition (sets the gain)

In [ ]:
gains = waterfall(*RUNS[0], muscles, xlim=XLIM)          # first condition sets the gain


### 3b · The other conditions — same gain

In [ ]:
for lab, (meta_, t_, sig_) in zip(LABELS[1:], RUNS[1:]):
    print(lab); waterfall(meta_, t_, sig_, muscles, xlim=XLIM, gains=gains)


### 3c · Overlay — all selected conditions on the same panels, same gain

In [ ]:
waterfall_overlay(RUNS, muscles=muscles, xlim=XLIM, gains=gains, labels=LABELS);


## 4 · Check — which peaks the peak-to-peak uses

Per muscle, every intensity, the max (red ▲) and min (blue ▼) inside the window that starts at
the picked latency. If a marker sits on the artifact or misses the wave, fix the pick (§2) or
`WINDOW_MS` / `OFFSET_MS` (§1).

### 4a · Each condition (files without picks are skipped)

In [ ]:
for lab, csv, (meta_, t_, sig_) in zip(LABELS, CSVS, RUNS):
    if not os.path.exists(_latfile(csv)):
        print(f"{lab}: no picks yet - skipped"); continue
    print(lab)
    plot_p2p_markers(meta_, t_, sig_, muscles, load_latency_csv(_latfile(csv)),
                     window_ms=WINDOW_MS, offset_ms=OFFSET_MS)


## 5 · Latency — across conditions

One panel per muscle, x = intensity. **Gray = Baseline, orange = VIB OFF, blue = VIB ON; ● solid =
left arm, ▲ dashed = right (vibrated) arm.** Vibration should show as the blue ▲ line leaving the
others while the blue ● stays with them. Trapezius is right-only.

In [ ]:
compare_per_muscle(CSVS, None, metric="latency", ylim=YLIM_LAT, labels=LABELS);


## 6 · Peak-to-peak — across conditions

Same layout, p2p in mV (window `[latency + OFFSET_MS, + WINDOW_MS]`). y-axis is per muscle —
biceps is ~10× the thenar — so compare gray vs orange within a panel, not heights across panels.

In [ ]:
compare_per_muscle(CSVS, None, metric="p2p", window_ms=WINDOW_MS, offset_ms=OFFSET_MS, labels=LABELS);
